<a href="https://colab.research.google.com/github/801-Hillside-Terrace/SMART-2026/blob/main/week3/Week3_SoftmaxRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Topics: Multiclass Classification, Softmax Regression/Multinomial Regression

Suppose that we now have $K>2$ classes.  So our target is now $y\in\{1,2,...,K\}.$  Logistic regression is no longer feasible as it only outputs a single probability.  With $K=3$ classes for example we need to model at least 2 of the 3 class probabilities to be able to make reasonable predictions.

Luckily, we can extend logistic regression with a slightly different function called the softmax:

$$\text{softmax}(z_k)=P(y_i=k|x)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}$$

The softmax function is bound between $0$ and $1$ and summing over all $K$ will always yield $1$ but we now need a $z$ for each class ($z_1$ through $z_K$) to model the probabilities:

$$z_k=w_k^\top x+b_k$$

We call the $z$'s logits.  $w_k$ is the $k$-th logit's vector of weights (coefficients - analogous to $\beta_1$ through $\beta_p$) and $b_k$ is the $k$-th logit's intercept term (like $\beta_0$).  So if we have $p$ features and $K$ classes, we now have $(p+1)× K$ parameters compared to the $p+1$ for logistic regression.

Now we need a loss function, we will end up with cross-entropy loss but extended from the fact that the target variable is no longer binomial but multinomial.  

$$\text{Let: } p_{ik}=P(y_i=k|x_i)\\
\mathbb{1}_{\{y_i=k\}}=\begin{cases} 1, & y_i=k \\ 0, &\text{o.w.}\end{cases}\\
\text{So }y_i\sim \text{Multinomial}(1, p_{i1},...,p_{iK}) \\
\text{The likelihood of a single observation } (y_i,x_i)\text{ is then: } \\
\prod_{k=1}^Kp_{ik}^{\mathbb{1}_{\{y_i=k\}}} \\
\text{The full likelihood is then: } L(W,b)=\prod_{i=1}^n\prod_{k=1}^Kp_{ik}^{\mathbb{1}_{\{y_i=k\}}} \\
\text{The log-likelihood is then: }\ell(W,b)=\sum_{i=1}^n\sum_{k=1}^K\mathbb{1}_{\{y_i=k\}}\ln(p_{ik}) \\
\text{So the cross-entropy loss is: }\mathcal{L}=-\sum_{i=1}^n\sum_{k=1}^K\mathbb{1}_{\{y_i=k\}}\ln(p_{ik})$$

We will use the same data, but now include the third species as a label (virginica).






In [9]:
#imports
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import csv
from sklearn.datasets import load_iris
# set seed
torch.manual_seed(801)

# load data
iris = load_iris()
X = iris['data']
y = iris['target']

# convert to tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long) # need integer labels


# split and standardize
def train_test_split(X, y, test_pct=0.2, seed=801):
    torch.manual_seed(seed)
    n = X.shape[0]
    perm = torch.randperm(n)

    split = int(n * (1 - test_pct))
    train_idx = perm[:split]
    test_idx = perm[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(dim=0, keepdim=True)
    std = X_train.std(dim=0, keepdim=True) + 1e-8

    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled

X_train, X_test, y_train, y_test = train_test_split(X, y, seed = 108)
X_train, X_test = standardize_train_test(X_train, X_test)

# check that we have roughly equal class representation in the training data
print(torch.unique(y_train, return_counts = True)[1])
y_train.unique() # note that in the data class 1 is encoded as 0, so y is in {0,1,2} here

tensor([40, 41, 39])


tensor([0, 1, 2])

We now have $p$ features and $K$ classes, so we have $p\times K$ weights (coefficients) and $K$ biases (intercepts).

In [10]:
p = X_train.shape[1] # number of features
print('Number of features: ', p)
K = torch.unique(y_train).shape[0]  # number of classes
print('Number of classes: ', K)

# need pxK weights and K biases
W = torch.randn((p, K)) * 0.01 # randomly initialize weights
W.requires_grad_() # require gradient
b = torch.zeros(K, requires_grad=True) # start biases at 0, require gradient

Number of features:  4
Number of classes:  3


Our forward function takes in the $n\times p$ matrix X and multiplies it by the $p\times K$ weight matrix before adding the bias vector to give the $n\times K$ $z_{ik}$ values.

In [11]:
def forward(X):
    # takes in X, gives nxK output of the logits (5th row first column is the 5th observation's logit for class 1)
    return X @ W + b   # nxK shape

The softmax function takes in the $z_{ik}$ values and exponentiates them before dividing that row's values by the sum along its columns.

In [12]:
def softmax(z):
    exp_z = torch.exp(z) # take exponential function of all logits (nxK shape)
    return exp_z / exp_z.sum(dim=1, keepdim=True) # normalize them by dividing by the sum of the K column entries

For the cross entropy loss we only care about the log-probabilities of the classes that actually correspond to each $y_i$.  IE if $y_7=1$ then we only add $-\ln(p_{7,1})$ to the loss for that observation.

In [13]:
def cross_entropy_loss(y, p):
    n = y.shape[0] # get length
    eps = 1e-8 # for numerical stability
    log_probs = torch.log(p + eps) # take log of all probabilities
    # we can index off y since classes are 0,1,2 and thus match the column indexes
    return -log_probs[torch.arange(n), y].mean() # take negative log of all probabilities from the column that matches their row's y value and average them

Here is the training loop, our parameters are the matrix of coefficients $W$ and the vector of biases $b$.  

In [14]:
optimizer = torch.optim.SGD([W, b], lr=0.3) # set up optimizer (gradient descent)

num_epochs = 10000 # number of epochs
for epoch in range(num_epochs):
    optimizer.zero_grad() # set gradients to zero

    logits = forward(X_train) # get logits (zs)
    probs = softmax(logits) # get probabilities
    loss = cross_entropy_loss(y_train, probs) # get loss

    loss.backward() # compute gradients
    optimizer.step() # update parameters

    if (epoch + 1) % 1000 == 0 or epoch == 0: # print training loss every 1000 iterations
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 1.0964
Epoch 1000, Loss: 0.0789
Epoch 2000, Loss: 0.0615
Epoch 3000, Loss: 0.0543
Epoch 4000, Loss: 0.0502
Epoch 5000, Loss: 0.0475
Epoch 6000, Loss: 0.0455
Epoch 7000, Loss: 0.0439
Epoch 8000, Loss: 0.0427
Epoch 9000, Loss: 0.0416
Epoch 10000, Loss: 0.0407


We predict by choosing the largest probability (which is the same as the largest logit).

In [15]:
def predict(X):
    with torch.no_grad(): # so gradients aren't tracked
        logits = forward(X) # get logits
        return torch.argmax(logits, dim=1) # get column index for largest logit (works since classes are 0, 1, 2 and thus are aligned with the column indexes)

Compute the accuracy by just counting the proportion of predicted $y$ that match the actual.

In [16]:
def accuracy(y_true, y_pred):
    return (y_true == y_pred).float().mean().item()

train_acc = accuracy(y_train, predict(X_train))
test_acc = accuracy(y_test, predict(X_test))

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 0.9916666746139526
Test Accuracy: 0.9666666388511658


Conceptual Questions:

1.  If $K=2$ and both models are fit such that they yield the same probability outputs, what difference exists between the softmax regression and logistic regression models?

2.  Why is choosing the class with the largest logit equivalent to choosing the class with the largest probability?

3.  What happens if the logit for just class $k$ increases?

4.  What happens to the probabilities if I add $100$ to every logit?

5.  What happens to the probabilities if I multiply every logit by $100$?

6.  If we have 7 features and 10 classes how many parameters do we have?